In [ ]:
import sys
print(sys.executable)

In [ ]:
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns 
import numpy as np
import json

In [ ]:
# read bigquery data into pandas dataframe
import pandas as pd

rfm_df = pd.read_gbq(
    """
    SELECT  *
    FROM `jr-data-training.nicole.rfm`
    --- LIMIT 10
  """,
    project_id="jr-data-training",
)

In [ ]:
rfm_df.info()

In [ ]:
rfm_df.head()

### Rewrite the RFM segment part

In [ ]:
# Create the quartiles scores

quantiles = rfm_df[['Recency', 'Frequency','Monetary']].quantile(q=[0.2,0.4,0.6,0.8])
quantiles = quantiles.to_dict()
    
def RScore(x,p,d):
    if x <= d[p][0.2]:
        return 5
    elif x <= d[p][0.4]:
        return 4
    elif x <= d[p][0.6]: 
        return 3
    elif x <= d[p][0.8]: 
        return 2
    else:
        return 1   
    
def FMScore(x,p,d):
    if x <= d[p][0.2]:
        return 1
    elif x <= d[p][0.4]:
        return 2
    elif x <= d[p][0.6]: 
        return 3
    elif x <= d[p][0.8]: 
        return 4
    else:
        return 5

rfm_df['R_score'] = rfm_df['Recency'].apply(RScore, args=('Recency',quantiles,))
rfm_df['F_score'] = rfm_df['Frequency'].apply(FMScore, args=('Frequency',quantiles,))
rfm_df['M_score'] = rfm_df['Monetary'].apply(FMScore, args=('Monetary',quantiles,))

In [ ]:
def join_rfm(x): return str(x['R_score']) + str(x['F_score']) + str(x['M_score'])

rfm_df['RFM_segment'] = rfm_df.apply(join_rfm, axis=1)
# Calculate RFM_Score
rfm_df['RFM_score'] = rfm_df[['R_score','F_score','M_score']].sum(axis=1)

In [ ]:
rfm_df['RFM_segment'].unique()

In [ ]:
# Create human friendly RFM labels
segt_map = {
    
    r'[1-2][1-2]': 'Hibernating',
    r'[1-2][3-4]': 'At risk',
    r'[1-2]5': 'Can\'t lose them',
    r'3[1-2]': 'About to sleep',
    r'33': 'Need attention',
    r'[3-4][4-5]': 'Loyal customers',
    r'41': 'Promising',
    r'51': 'New customers',
    r'[4-5][2-3]': 'Potential loyalists',
    r'5[4-5]': 'Champions'
}
# rfm['Segment'] = rfm['R'].map(str) + rfm['F'].map(str)+ rfm['M'].map(str)
rfm_df['Segment'] = rfm_df['R_score'].map(str) + rfm_df['F_score'].map(str)
rfm_df['Segment'] = rfm_df['Segment'].replace(segt_map, regex=True)
# Create some human friendly labels for the scores
rfm_df['Score'] = 'Green'
rfm_df.loc[rfm_df['RFM_score']>5,'Score'] = 'Bronze' 
rfm_df.loc[rfm_df['RFM_score']>7,'Score'] = 'Silver' 
rfm_df.loc[rfm_df['RFM_score']>9,'Score'] = 'Gold' 
rfm_df.loc[rfm_df['RFM_score']>10,'Score'] = 'Platinum'
# List the head of the table to view the 
rfm_df.head(5)

### Write to GBQ

In [ ]:
import pandas_gbq

In [ ]:
project_id = "jr-data-training"
table_id = 'nicole.rfm_new'

pandas_gbq.to_gbq(rfm_df,table_id, project_id=project_id)